In [1]:
import os
import sys
import time
import argparse
import numpy as np
import torch
import config as cfg

from model import MyNet, ResNet18
from dataset import get_dataloader
from utils import write_csv
device = 'cuda'

model_path = '/mnt/20F408ADF408876E/114_2/computer_vision/homeworks/hw2/p2/experiment/resnet18_2026_04_04_17_26_04_default/model/model_best.pth'
model = ResNet18()
model.load_state_dict(torch.load(model_path, 
                                    map_location=torch.device('cpu')))

model = model.to(device)


In [2]:
unlabel_dataset_path = '/mnt/20F408ADF408876E/114_2/computer_vision/hw2_data/p2_data/unlabel'
data_loader = get_dataloader(
    unlabel_dataset_path,
    batch_size=cfg.batch_size, split='test',
    return_img_names= True
)


Number of test images is 30000


In [ ]:
from tqdm import tqdm
threshold = 0.9
filenames = []
labels = []
model.eval()
# print('total:', len(data_loader), '\n\n')
with torch.no_grad():
    for batch, data in tqdm(enumerate(data_loader)):
        images = data['images'].to(device)
        img_names = data['image_names']

        pred = model(images)
        pred_n = torch.softmax(pred, dim=1)

        max_probs, pseudo_labels = torch.max(pred_n, dim=1)

        indexes = torch.where(max_probs >= threshold)[0]

        for idx in indexes:
            filenames.append(img_names[idx])
            labels.append(pseudo_labels[idx].item())
        # break

total: 235 




235it [00:09, 24.41it/s]


In [8]:
import json
output_path = './pseudo_labels.json'
with open(output_path, 'w') as f:
    json.dump(
        {
            'filenames': filenames,
            'labels': labels
        } ,f
    )

In [9]:
val_path = '/mnt/20F408ADF408876E/114_2/computer_vision/hw2_data/p2_data/val'

In [12]:
val_loader = get_dataloader(
    dataset_dir = unlabel_dataset_path, 
    # dataset_dir = val_path,
    batch_size=cfg.batch_size, 
    split='val', 
    unlabel_annotation_path='./pseudo_labels.json'
)

Number of val images is 12319


In [ ]:
model.eval()
with torch.no_grad():
    val_start_time = time.time()
    cond_val_correct = 0.0
    val_n_total = 0
    for batch, data in tqdm(enumerate(val_loader)):
        # Data loading. (batch_size, 3, 32, 32), (batch_size)
        images, labels = data['images'].to(device), data['labels'].to(device)
        # Forward pass. input: (batch_size, 3, 32, 32), output: (batch_size, 10)
        pred = model(images)
        pred_n = torch.softmax(pred, dim=1)

        max_probs, pseudo_labels = torch.max(pred_n, dim=1)
        indexes = torch.where(max_probs >= threshold)[0]
        val_n_total += len(indexes)
        cond_val_correct += torch.sum( pseudo_labels[indexes] == labels[indexes] )

        # Calculate loss.
        # loss = criterion(pred, labels)
        # val_correct += torch.sum(torch.argmax(pred, dim=1) == labels)
        # val_loss += loss.item()

    #############################################################
    # TODO:                                                     #
    # Finish forward part in validation, you can refer to the   #
    # training part.                                            #
    #                                                           #
    # NOTE:                                                     #
    # You don't have to update parameters, just record the      #
    # accuracy and loss.                                        #
    #############################################################

    ######################### TODO End ##########################

# Print validation result
val_time = time.time() - val_start_time
cond_val_acc = cond_val_correct / val_n_total
print('num of samples:', val_n_total)
print("conditional val correct:", cond_val_correct)
print(f'conditional val correct: Acc: {cond_val_acc:.5f}')



num of samples: 12319
conditional val correct: tensor(12319., device='cuda:0')
conditional val correct: Acc: 1.00000
